# Step 2: Process Features
Ce notebook évalue sur chaque segment du réseaux chaque attribut. La méthode de comptabilisation de chaque attribut est prédéfinie dans le fichier excel attributs_info.

## Traitement des attributs

In [1]:
%load_ext autoreload
%autoreload 2


import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point
from matplotlib import pyplot as plt

import os, time
import numpy as np
import geopandas as gpd
import shapely


import os

# from method_a_buffer import extract_buffer_feature
from feature_to_network import extract_buffer_feature
from feature_query import make_feature_query

import geopandas as gpd
import folium
from folium import Choropleth, CircleMarker, GeoJson
import osmnx as ox
import branca.colormap as cm
import rasterio
# Display the map in the notebook
from IPython.display import display
# Display all columns in a dataframe
pd.set_option('display.max_columns', None)


operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters : GG or GE, bike or walk")
territory = 'GG' # GG or GE
network = "bike"  # walk or bike

if territory == 'GG':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GG/step-1'
    output_step2_path='../../Data/output/GG/step-2'
    output_step3_path='../../Data/output/GG/step-3'


    save_path = '../../Data/output/GG/step-2'

    attribute_source_path = 'source_path_GG'
    file_name = 'file_name_GG'  # column name in attributs_info excel

    how_name = 'how_GG'
    geometry_type_name = 'geometry_type_GG'
    value_column_name = 'value_column_GG'

if territory == 'GE':
    input_file_path = '../../Data/input'
    output_step1_path='../../Data/output/GE/step-1'
    output_step2_path='../../Data/output/GE/step-2'
    output_step3_path='../../Data/output/GE/step-3'


    save_path = '../../Data/output/GE/step-2'

    attribute_source_path = 'source_path_GE'
    file_name = 'file_name_GE'  # column name in attributs_info excel
    how_name = 'how_GE'
    geometry_type_name = 'geometry_type_GE'
    value_column_name = 'value_column_GE'



# Load segments GeoDataFrame (with 'segment_id')
print("Loading pedestrian segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net["geometry"] = segmented_net["geometry"].apply(lambda geom: geom.buffer(0) if not geom.is_valid else geom)
segmented_net = segmented_net.to_crs(operation_crs)


Set parameters : GG or GE, bike or walk
Loading pedestrian segments...


In [2]:

# Import des attributs
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info[attributs_info['include_in_index'] != False]

attributs_info

,Class,meta_attribute,attribute,include_in_index,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip,filter_column,filter_values,crs,save_format,Unnamed: 32,commentaire
0,attractivite,amenite,amenite,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,75.0,favorable,point,count,NaN,point,count,NaN,4.0,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
2,attractivite,connectivite,connectivite,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.8,0.5,10.0,favorable,line,sum,conn_branching_in_buffer,line,sum,conn_branching_in_buffer,7.0,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
3,attractivite,pente,pente,True,opendata,attributs/GE/topographie,pente_2056.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/topographie,pente_2056.tif,prefer_opendata,0.3,0.5,10.0,defavorable,line,sum,slope_pct_abs_smoothed,line,sum,slope_pct_abs_smoothed,75.0,filtered,1,2056,"parquet, csv, gpkg",NaN,top
4,confort,lac_cours_deau,eau,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.4,0.4,50.0,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
5,confort,temperature,temperature,True,opendata,attributs/GE/temperature,20250629_102250.LST.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,20250629_102250.LST.tif,prefer_opendata,0.4,0.4,250.0,defavorable,point,raster,temperature,point,raster,temperature,40.0,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
6,confort,air,air,True,opendata,attributs/GE/air,NO2_Extraction_2025.tif,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/air,NO2_Extraction_2025.tif,prefer_opendata,0.5,0.5,750.0,defavorable,point,raster,no_2,point,raster,no_2,NaN,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
7,confort,landuse_env,alentours,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.3,0.3,50.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
8,confort,canopee,canopee,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.5,0.5,25.0,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,NaN,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
9,equipement,stationnement_velo,stationnement_velo,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.7,0.7,30.0,favorable,point,sum,score,point,sum,score,5.0,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN
10,equipement,borne_reparation,borne_reparation,True,osm,attributs/GE/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,prefer_opendata,0.6,0.6,50.0,favorable,point,count,NaN,point,count,NaN,NaN,filtered,1,2056,"parquet, csv, gpkg",NaN,NaN


V1 Original

In [3]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for _, row in attributs_info.iterrows():
#     if row['include_in_index']:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
#         feature_query_expr = make_feature_query(
#         row.get('filter_column'),
#         row.get('filter_values')
#     )

#         # Charger la couche attribut depuis gpd_attributs
#         print(f"Chargement de la couche: {attribute_name}")
#         attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#         attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#         # Appliquer la méthode
        
#         if method == "A": # Buffer feature extraction
#             print(f"Applying method {method} for attribute {attribute_name}...")
#             attribute_df = extract_buffer_feature(
#                 segments_gdf = segmented_net,
#                 feature_gdf = attribute_gdf,
#                 feature_name=attribute_name,
#                 geom_kind=geometry_type,
#                 buffer_radius=buffer_size,
#                 how=how,
#                 value_column=value_column,
#                 crs_meter_epsg=operation_crs,
#                 feature_query  = feature_query_expr,
#             )
#             print(f"Buffer feature extracted for {attribute_name} with method {method}")
        
#                 # Debug duplicates
#             if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                 print("\nDEBUG: Found duplicate segment assignments")
#                 dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                 print(f"Number of segments with multiple zones: {len(dupes['segment_id'].unique())}")
            
#                 # Keep only the first occurrence for each segment_id
#                 attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                 print("Dropped duplicates, keeping first occurrence")
#             print(f"Spatial join computed for {attribute_name} with method {method}") 
        
#         # Ajoute d'autres méthodes si besoin
        
#         # Ajouter la colonne au GeoDataFrame principal
#         segmented_net[f'{attribute_name}_{method}_{buffer_size}'] = attribute_df[attribute_name].fillna(0)
#         print(f"Attribute {attribute_name} added to segmented_net.")

# # Sauvegarder
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)


V2

In [4]:
# # Garder uniquement les colonnes 'segment_id', 'vitesse', et 'geometry' dans segmented_net
# segmented_net = segmented_net[['geometry', 'segment_id', 'length']].copy()

# '''# --- ADD THIS BLOCK ---
# # Sample a subset of the network for faster testing
# sample_fraction = 0.7  # 70% of all segments, adjust as needed
# segmented_net = segmented_net.sample(frac=sample_fraction, random_state=42)

# print(f"Testing on a sample of {len(segmented_net)} segments out of the total network.")
# # ----------------------'''

# print('Boucle sur chaque attribut... peut prendre du temps (30 mn)')

# # Boucle sur le derniers attributs, first 3 only for testing
# for idx, row in attributs_info.iterrows():
#     if row['include_in_index']:
#         attribute_name = row['attribute']
#         method = row['method']
#         how = row['how']
#         value_column = row['value_column']
#         buffer_size = row['buffer_size']
#         geometry_type = row['geometry_type']
        
#         try:
#             feature_query_expr = make_feature_query(
#                 row.get('filter_column'),
#                 row.get('filter_values')
#             )

#             # Charger la couche attribut depuis gpd_attributs
#             print(f"\n[{idx+1}/{len(attributs_info)}] Chargement de la couche: {attribute_name}")
#             attribute_gdf = gpd.read_parquet(f"../../Data/output/step-2/parquet_attributs/{attribute_name}.parquet")
#             print(f"  → {len(attribute_gdf)} features chargées")
#             attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

#             # Appliquer la méthode
#             if method == "A": # Buffer feature extraction
#                 print(f"  → Applying method {method} (how={how}, buffer={buffer_size}m)...")
#                 attribute_df = extract_buffer_feature(
#                     segments_gdf = segmented_net,
#                     feature_gdf = attribute_gdf,
#                     feature_name=attribute_name,
#                     geom_kind=geometry_type,
#                     buffer_radius=buffer_size,
#                     how=how,
#                     value_column=value_column,
#                     crs_meter_epsg=operation_crs,
#                     feature_query  = feature_query_expr,
#                 )
#                 print(f"  → Buffer feature extracted")
            
#                 # Debug duplicates
#                 if attribute_df[attribute_df.duplicated('segment_id')].shape[0] > 0:
#                     print("  → WARNING: Found duplicate segment assignments")
#                     dupes = attribute_df[attribute_df.duplicated('segment_id', keep=False)]
#                     print(f"     Segments with duplicates: {len(dupes['segment_id'].unique())}")
                
#                     # Keep only the first occurrence for each segment_id
#                     attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')
#                     print("  → Duplicates dropped (keeping first)")
            
#             # Ajoute d'autres méthodes si besoin
            
#             # Ajouter la colonne au GeoDataFrame principal
#             col_name = f'{attribute_name}_{method}_{buffer_size}'
#             segmented_net[col_name] = attribute_df[attribute_name].fillna(0)
#             print(f"  ✓ Attribute {attribute_name} added to segmented_net as '{col_name}'")
            
#         except Exception as e:
#             print(f"  !!! ERROR processing {attribute_name}: {e}")
#             import traceback
#             traceback.print_exc()
#             # Continue avec l'attribut suivant au lieu de crasher
#             continue

# # Sauvegarder
# print("\n" + "="*60)
# print("Saving step 2 features to parquet...")
# segmented_net.to_crs(target_crs).to_parquet(os.path.join(output_step2_path, "step2_features.parquet"), index=False)
# print(f"✓ Saved {len(segmented_net)} segments with {len(segmented_net.columns)} columns")
# print("="*60)

V3

In [5]:
import os, time
import numpy as np
import geopandas as gpd
import shapely
from feature_query import make_feature_query  # ta fonction inline

# ========= Réglages de performance =========
BUFFER_RESOLUTION = 2           # précision suffisante
UNION_LINES_FOR_LENGTH_RATIO = True  # éviter le double comptage
ROUND_DECIMALS = 3              # arrondi à 3 décimales pour tous les floats

# ========= Préparer le réseau (CRS + colonnes utiles) =========
if segmented_net.crs is None or segmented_net.crs.to_epsg() != operation_crs:
    segmented_net = segmented_net.to_crs(operation_crs)

segmented_net = segmented_net[['segment_id', 'geometry', 'length']].copy()
segmented_net = segmented_net.drop_duplicates('segment_id')
segmented_net = segmented_net.set_index('segment_id', drop=False)

print('Boucle sur chaque attribut (méthode définie dans la table attributaire)…')


for _, row in attributs_info.iterrows():
    if not row.get('include_in_index', False):
        continue

    attribute_name = row['attribute']
    how            = row[how_name]                     # presence / count / mean / sum / area_ratio / length_ratio / raster
    value_column   = row.get(value_column_name, None)
    buffer_size    = float(row['buffer_size'])
    geometry_type  = str(row[geometry_type_name]).lower().strip()

    print(f"\n→ {attribute_name} | how={how}, geom={geometry_type}, buffer={buffer_size}m")
    t0 = time.perf_counter()

    # 1) Charger la couche attribut + CRS
    attr_path = f"{output_step2_path}/parquet_attributs/{attribute_name}.parquet"
    attribute_gdf = gpd.read_parquet(attr_path)
    if attribute_gdf.crs != segmented_net.crs:
        attribute_gdf = attribute_gdf.to_crs(segmented_net.crs)

    # 2) Filtrer via make_feature_query (inline)
    expr = make_feature_query(row.get('filter_column'), row.get('filter_values'))
    if expr:
        try:
            attribute_gdf = attribute_gdf.query(expr)
        except Exception as e:
            print(f"  ⚠️ Filtre ignoré ({expr}) : {e}")

    # 3) Nettoyage géométries
    attribute_gdf = attribute_gdf[
        attribute_gdf.geometry.notna() & ~attribute_gdf.geometry.is_empty
    ].copy()
    if not attribute_gdf.geometry.is_valid.all():
        attribute_gdf['geometry'] = shapely.make_valid(attribute_gdf.geometry.values)

    print(f"  Taille attribut: {len(attribute_gdf):,} | chargement+filtre: {time.perf_counter()-t0:.1f}s")

    # 5) value_column peut être NaN → None
    if isinstance(value_column, float) and np.isnan(value_column):
        value_column = None

    # 6) Extraction (toujours méthode A)
    t1 = time.perf_counter()
    attribute_df = extract_buffer_feature(
        segments_gdf        = segmented_net.reset_index(drop=True),
        feature_gdf         = attribute_gdf,
        feature_name        = attribute_name,
        geom_kind           = geometry_type,
        how                 = how,
        buffer_radius       = buffer_size,
        value_column        = value_column,
        crs_meter_epsg      = operation_crs,
        predicate           = row.get('predicate', None),
        feature_query       = None,  # déjà filtré
        segment_length_col  = 'length',
        zero_for_missing    = True,
        raster_stats        = row.get('raster_stats', 'mean'),
        buffer_resolution   = BUFFER_RESOLUTION,
    )
    print(f"  ✓ extract_buffer_feature: {time.perf_counter()-t1:.1f}s — {attribute_df.shape[0]} lignes")

    # 7) Sécurité unicité
    if attribute_df.duplicated('segment_id').any():
        ndup = attribute_df[attribute_df.duplicated('segment_id', keep=False)]['segment_id'].nunique()
        print(f"  ⚠️ Duplicates détectés pour {ndup} segments → on garde le premier")
        attribute_df = attribute_df.drop_duplicates('segment_id', keep='first')

    # 8) Assignation + arrondi
    attribute_df = attribute_df.set_index('segment_id')
    colname = f"{attribute_name}_{int(buffer_size)}"  # plus de “_A_” ici

    if colname not in segmented_net.columns:
        segmented_net[colname] = 0.0

    ix = segmented_net.index.intersection(attribute_df.index)
    vals = attribute_df.loc[ix, attribute_name].to_numpy(dtype='float64')
    segmented_net.loc[ix, colname] = np.around(vals, ROUND_DECIMALS)

    print(f"  → Colonne ajoutée: {colname} (arrondie à {ROUND_DECIMALS} décimales)")

# 9) Sauvegarde (et arrondi global final)
print("\nSaving step 2 features to parquet…")
if segmented_net.crs != target_crs:
    segmented_net = segmented_net.to_crs(target_crs)

float_cols = segmented_net.select_dtypes(include=['float', 'float32', 'float64']).columns
segmented_net[float_cols] = segmented_net[float_cols].astype('float64').round(ROUND_DECIMALS)

out_path = os.path.join(output_step2_path, "step2_features.parquet")
segmented_net.to_parquet(out_path, index=False)
print(f"✔️ Enregistré: {out_path}")

Boucle sur chaque attribut (méthode définie dans la table attributaire)…

→ amenite | how=count, geom=point, buffer=75.0m
  Taille attribut: 30,044 | chargement+filtre: 0.1s
  ✓ extract_buffer_feature: 6.8s — 723151 lignes
  → Colonne ajoutée: amenite_75 (arrondie à 3 décimales)

→ connectivite | how=sum, geom=line, buffer=10.0m
  Taille attribut: 723,151 | chargement+filtre: 1.1s
  ✓ extract_buffer_feature: 6.7s — 723151 lignes
  → Colonne ajoutée: connectivite_10 (arrondie à 3 décimales)

→ pente | how=sum, geom=line, buffer=10.0m
  Taille attribut: 723,151 | chargement+filtre: 1.2s
  ✓ extract_buffer_feature: 6.9s — 723151 lignes
  → Colonne ajoutée: pente_10 (arrondie à 3 décimales)

→ eau | how=presence, geom=polygon, buffer=50.0m
  Taille attribut: 21,997 | chargement+filtre: 0.3s
  ✓ extract_buffer_feature: 360.3s — 723151 lignes
  → Colonne ajoutée: eau_50 (arrondie à 3 décimales)

→ temperature | how=raster, geom=point, buffer=250.0m
  Taille attribut: 406,794 | chargement+fil

In [6]:

segmented_net.head(20)


,segment_id,geometry,length,amenite_75,connectivite_10,pente_10,eau_50,temperature_250,air_750,alentours_50,canopee_25,stationnement_velo_30,borne_reparation_50,location_100,sens_inverse_10,service_velo_25,parking_abris_25,piste_10,bande_10,revetement_10,giratoire_20,tourner_droite_25,eclairage_10,zone_apaisee_10,vitesse_motorisee_10,conflit_md_25,accident_100
segment_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
000000,000000,"LINESTRING (6.15792 46.20527, 6.15782 46.20537...",33.141,32.0,5.000,93.345,1.0,30.821,36.316,0.346,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,88.0,0.0,0.0,1.0,0.0,1.0,94.0,20.0
000001,000001,"LINESTRING (6.15767 46.20551, 6.15777 46.20542...",33.141,32.0,5.000,93.345,1.0,30.821,36.316,0.346,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,88.0,0.0,0.0,1.0,0.0,1.0,94.0,20.0
000002,000002,"LINESTRING (5.95491 46.21124, 5.95495 46.21126...",50.000,0.0,1.850,26.999,0.0,41.742,31.913,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000003,000003,"LINESTRING (5.95538 46.21155, 5.95577 46.21181...",50.000,0.0,0.706,15.361,0.0,41.586,31.913,0.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000004,000004,"LINESTRING (5.95584 46.21186, 5.95624 46.21222)",50.000,0.0,0.650,13.779,0.0,41.494,31.913,0.343,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000005,000005,"LINESTRING (5.95624 46.21222, 5.95663 46.21258)",50.000,0.0,0.762,12.407,0.0,41.302,31.913,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000006,000006,"LINESTRING (5.95663 46.21258, 5.95686 46.21278)",28.789,0.0,0.762,11.488,0.0,41.134,31.913,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000007,000007,"LINESTRING (5.95686 46.21278, 5.95646 46.21242)",50.000,0.0,0.900,14.014,0.0,41.130,31.913,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
000008,000008,"LINESTRING (5.95646 46.21242, 5.95607 46.21207)",50.000,0.0,0.725,12.774,0.0,41.490,31.913,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
#import step2_features and convert to gpkg for QGIS use
segmented_net = gpd.read_parquet(os.path.join(output_step2_path, "step2_features.parquet"))
segmented_net.to_file(os.path.join(output_step2_path, "step2_features.gpkg"), driver="GPKG")
